## T Test and Bonferroni

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

SCI_FILE   = Path("pr_metrics_all_sci.csv")
NONSCI_FILE = Path("pr_metrics_all_non_sci.csv")
OUT_TEST   = Path("ttests_bonferroni_by_repo.csv")

ALPHA = 0.05
M     = 6  # six metrics

METRICS = [
    "Time to Merge (days)",
    "Number of Unique Reviewers",
    "Total Discussion Comments",
    "Commits After First Review",
    "Time Between First and Last Comments (days)",
    "Requested Changes",
]

def welch_ttest(x: np.ndarray, y: np.ndarray):
    from scipy.stats import ttest_ind
    res = ttest_ind(x, y, equal_var=False, nan_policy="omit")
    nx = np.isfinite(x).sum()
    ny = np.isfinite(y).sum()
    vx = np.nanvar(x, ddof=1)
    vy = np.nanvar(y, ddof=1)
    num = (vx/nx + vy/ny)**2
    den = (vx**2 / (nx**2*(nx-1))) + (vy**2 / (ny**2*(ny-1)))
    df = num/den if den > 0 else np.nan
    return float(res.statistic), float(res.pvalue), float(df), nx, ny

if __name__ == "__main__":
    sci = pd.read_csv(SCI_FILE)
    nonsci = pd.read_csv(NONSCI_FILE)

    sci["SK_label"] = 1
    nonsci["SK_label"] = 0

    df = pd.concat([sci, nonsci], ignore_index=True)

    results = []

    for repo in df["repo"].unique():
        df_repo = df[df["repo"] == repo]

        for metric in METRICS:
            logm = f"log1p {metric}"
            if logm not in df_repo.columns:
                print(f"Missing {logm} in {repo}, skipping.")
                continue

            g0 = df_repo.loc[df_repo["SK_label"] == 0, logm].to_numpy(dtype=float)
            g1 = df_repo.loc[df_repo["SK_label"] == 1, logm].to_numpy(dtype=float)

            if (len(g0) < 2) or (len(g1) < 2):
                continue

            t_stat, p_raw, df_welch, n0, n1 = welch_ttest(g0, g1)
            p_bonf = min(p_raw * M, 1.0)

            orig0 = pd.to_numeric(df_repo.loc[df_repo["SK_label"] == 0, metric], errors="coerce")
            orig1 = pd.to_numeric(df_repo.loc[df_repo["SK_label"] == 1, metric], errors="coerce")

            results.append({
                "repo": repo,
                "metric": metric,
                "tested_on": "log1p",
                "n_non_sci": int(n0),
                "n_sci": int(n1),
                "mean_non_sci_orig": float(np.nanmean(orig0)) if len(orig0) else np.nan,
                "mean_sci_orig": float(np.nanmean(orig1)) if len(orig1) else np.nan,
                "t_stat": t_stat,
                "df_welch": df_welch,
                "p_raw": p_raw,
                "p_bonferroni": p_bonf,
                "significant_at_alpha/Bonferroni": bool(p_bonf < ALPHA),
                "bonferroni_alpha": ALPHA / M
            })

    pd.DataFrame(results).to_csv(OUT_TEST, index=False)
    print(f"Wrote {OUT_TEST.resolve()} (tests={len(results)})")

Wrote /content/ttests_bonferroni_by_repo.csv (tests=18)
